# ASE GA

## Introduction
This notebook demonstrates the basic framework and workflow of the genetic alogorithm (GA) implementation in the ASE, finally leading to help us build more adapted GA for our specific tasks.

### The possible applications of ASE GA 

* small cluster on/in support material in:
> L. B. Vilhelmsen and B. Hammer    
> [A genetic algorithm for first principles global structure optimization of supported nano structures](https://doi.org/10.1063/1.4886337)    
> The Journal of chemical physics, Vol. 141 (2014), 044711

* medium sized alloy clusters:
> S. Lysgaard, D. D. Landis, T. Bligaard and T. Vegge       
> [Genetic Algorithm Procreation Operators for Alloy Nanoparticle Catalysts](https://doi.org/10.1007/s11244-013-0160-9)      
> Topics in Catalysis, Vol 57, No. 1-4, pp. 33-39, (2014)     

* mixed metal ammines for ammonia storage:
> P. B. Jensen, S. Lysgaard, U. J. Quaade and T. Vegge    
> [Designing Mixed Metal Halide Ammines for Ammonia Storage Using Density Functional Theory and Genetic Algorithms](https://doi.org/10.1039/C4CP03133D)    
> Physical Chemistry Chemical Physics, Vol 16, No. 36, pp. 19732-19740, (2014)    

* Crystal structures searches
> M. Van den Bossche, H. Grönbeck, and B. Hammer        
> [Tight-Binding Approximation-Enhanced Global Optimization](https://doi.org/10.1021/acs.jctc.8b00039)      
> J. Chem. Theory Comput. 2018, 14, 2797−2807

### Our goal

* To seach the potential energy surface (PES) of cluster, surface, and adsorbate systems efficiently and find the global minimum structure.
* Based on the global minimum strcture, to explore the grand canonical ensemble of structures under different temperature and pressure conditions.

## Surface/interface GA based on ASE

### 1. Define the basic properties of the structures

* A list of atomic numbers for the structure to be optimized
* A cell box or a substrate to randomly distribute the atoms

In [1]:
# Build the substrate
from ase.build import fcc111, molecule
from ase.constraints import FixAtoms

substrate = fcc111('Cu', size=(4, 4, 4), vacuum=10.0, orthogonal=True)
substrate.set_constraint(FixAtoms(
    [atom.index for atom in substrate if atom.position[2] < substrate.positions[:, 2].mean() + 0.1])
    )

In [2]:
# Define the space to place the atoms
cell = substrate.get_cell()
pos = substrate.get_positions()

# The coner position of the box
p0 = [0, 0, pos[:, 2].max() + 1.2]
v1 = cell[0] * 0.8
v2 = cell[1] * 0.8
v3 = [0, 0, 3.0]  # height of the box

In [3]:
# Define the placed atoms or molecules
# If atoms, it can be randomly placed and optimized directly
# else, if molecules, it should set the tags and keep the native of the molecules
atom_numbers = [6, 8] * 10

# Define the bond length judgement criteria during the whole GA process
from ase_ga_official.utilities import closest_distances_generator, get_all_atom_types
unique_atom_types = get_all_atom_types(substrate, atom_numbers)
bl_min = closest_distances_generator(unique_atom_types, 0.8)

In [13]:
# Create the starting population

# StartGenerator focus on the following functionality:
#     1. Define the base structure(slab or None), box to place atoms/molecules, added atom numbers/molecules

from ase_ga_official.startgenerator import StartGenerator

In [14]:
sg = StartGenerator(
    slab = substrate,
    blocks = [
    ('CO', 3)
    ],
    blmin=bl_min,
    number_of_variable_cell_vectors=0
)

In [10]:
sg.get_new_candidate(maxiter=100)

In [17]:
from ase.visualize import view

view(sg.slab)

<Popen: returncode: None args: ['/Users/ychao/Documents/conda/env/ase/bin/py...>